# Real-Pipeline Virtual Geometry Test for 3D-UNet Surrogate

Create one new virtual geometry and tests the trained surrogate against the same physical label-generation pipeline used for the training dataset.




In [ ]:
from pathlib import Path
import json
import math
import shutil
import numpy as np
import matplotlib.pyplot as plt
import torch

from south_kensington_jupyter import SouthKensingtonConfig
from south_kensington_temperature_3d import Temperature3DScenarioConfig, run_temperature_3d_pipeline
from temperature_surrogate_3d import (
    Temperature3DCase,
    TemperatureSurrogateDataset,
    SurrogateStats,
    UNet3D,
)

BUNDLE_ROOT = Path.cwd()
if not (BUNDLE_ROOT / 'south_kensington_temperature_3d.py').exists():
    BUNDLE_ROOT = Path('velocity/temperature_field_bundle').resolve()

RUN_DIR = BUNDLE_ROOT / 'outputs' / 'temperature_3d_unet_surrogate'
CHECKPOINT_PATH = RUN_DIR / 'best_temperature_3d_unet_surrogate.pt'
TEST_ROOT = BUNDLE_ROOT / 'outputs' / 'virtual_geometry_real_pipeline_test_complex_v2'
STATIC_DIR = TEST_ROOT / 'static_virtual_complex_v2'
CASE_DIR = TEST_ROOT / 'cases' / 'case_virtual_complex_v2_v035_t01'
VELOCITY_CACHE_DIR = TEST_ROOT / 'velocity_cache' / 'virtual_complex_v2_v035'
VELOCITY_DIGIT_DIR = TEST_ROOT / 'velocity_digit_outputs' / 'virtual_complex_v2_v035'

TEST_ROOT.mkdir(parents=True, exist_ok=True)
STATIC_DIR.mkdir(parents=True, exist_ok=True)
CASE_DIR.mkdir(parents=True, exist_ok=True)
VELOCITY_CACHE_DIR.mkdir(parents=True, exist_ok=True)
VELOCITY_DIGIT_DIR.mkdir(parents=True, exist_ok=True)

print('Bundle root:', BUNDLE_ROOT)
print('Checkpoint:', CHECKPOINT_PATH)
print('Test root:', TEST_ROOT)


## Create a Virtual Static Geometry


In [ ]:
Y_DIM = 112
X_DIM = 112
GRID_RESOLUTION_M = 4.0
HEIGHT_SCALE_M = 4.0
Z_DIM = 10
EMBED_PAD_Y = 8
EMBED_PAD_X = 8

building_mask = np.zeros((Y_DIM, X_DIM), dtype=np.uint8)
building_height = np.zeros((Y_DIM, X_DIM), dtype=np.float32)
land_cover = np.full((Y_DIM, X_DIM), 50, dtype=np.uint8)  # urban fabric by default

yy, xx = np.mgrid[0:Y_DIM, 0:X_DIM]


def add_rotated_rect(center_y, center_x, length, width, angle_deg, height_m):
    angle = np.deg2rad(angle_deg)
    dy = yy - float(center_y)
    dx = xx - float(center_x)
    along = dx * np.cos(angle) + dy * np.sin(angle)
    across = -dx * np.sin(angle) + dy * np.cos(angle)
    rect = (np.abs(along) <= length / 2.0) & (np.abs(across) <= width / 2.0)
    building_mask[rect] = 1
    building_height[rect] = np.maximum(building_height[rect], float(height_m))


def add_axis_rect(y0, y1, x0, x1, height_m):
    add_rotated_rect((y0 + y1) / 2, (x0 + x1) / 2, x1 - x0, y1 - y0, 0.0, height_m)

# A larger, more varied virtual urban layout. It fills most of the embedded
# xy domain and includes rotated blocks, long bars, small buildings, and towers.
add_rotated_rect(15, 18, 20, 8, -18, 22.0)
add_rotated_rect(18, 44, 28, 7, 11, 34.0)
add_rotated_rect(15, 82, 25, 10, -12, 18.0)
add_rotated_rect(32, 24, 16, 16, 0, 42.0)
add_rotated_rect(36, 62, 18, 12, 28, 30.0)
add_rotated_rect(33, 93, 24, 9, 16, 24.0)
add_rotated_rect(53, 17, 28, 8, 37, 26.0)
add_rotated_rect(55, 45, 14, 26, -7, 46.0)
add_rotated_rect(56, 78, 30, 8, -24, 20.0)
add_rotated_rect(78, 19, 18, 26, -10, 28.0)
add_rotated_rect(82, 50, 30, 12, 18, 38.0)
add_rotated_rect(84, 83, 20, 26, -18, 44.0)
add_rotated_rect(102, 33, 24, 7, -32, 16.0)
add_rotated_rect(101, 68, 28, 8, 9, 26.0)

# A few compact taller towers.
add_rotated_rect(43, 37, 8, 8, 0, 52.0)
add_rotated_rect(70, 67, 9, 9, 0, 56.0)
add_rotated_rect(25, 71, 7, 7, 0, 48.0)
add_rotated_rect(74, 35, 7, 7, 0, 44.0)

# Add some cooler parks/open ground corridors so the thermal solver sees mixed land use.
land_cover[7:23, 91:109] = 10
land_cover[88:108, 5:24] = 10
land_cover[46:67, 55:69] = 10
land_cover[21:31, 6:106] = 30
land_cover[66:75, 8:104] = 30
land_cover[:, 51:59] = 30
land_cover[building_mask > 0] = 50

np.save(STATIC_DIR / 'building_mask.npy', building_mask)
np.save(STATIC_DIR / 'building_height.npy', building_height)
np.save(STATIC_DIR / 'land_cover.npy', land_cover)

virtual_grid = {
    'resolution_m': GRID_RESOLUTION_M,
    'shape': [Y_DIM, X_DIM],
    'origin_x_m': 0.0,
    'origin_y_m': 0.0,
    'crs': 'virtual-local-grid',
}
(STATIC_DIR / 'grid.json').write_text(json.dumps(virtual_grid, indent=2))

print('Virtual static dir:', STATIC_DIR)
print('Building coverage:', round(100 * building_mask.mean(), 1), '%')
print('Max building height m:', float(building_height.max()))
print('Land cover counts:', {int(v): int((land_cover == v).sum()) for v in np.unique(land_cover)})
print('Expected embedded output xy is approximately:', (Y_DIM + 2 * EMBED_PAD_Y, X_DIM + 2 * EMBED_PAD_X))


## Configure the Existing DIGIT + 3D Temperature Pipeline



In [ ]:
# Existing DIGIT velocity settings, matching the 48-case dataset scale.
# Note: the current DIGIT pipeline does not expose arbitrary inlet wind direction.
# Wind-direction variation in the current dataset was represented by rotating geometry.
# Here we keep the pipeline unchanged and use one virtual inlet-flow case.
INLET_FLOW = 0.35
REFERENCE_WIND_DIRECTION_DEG = 268.41

velocity_config = SouthKensingtonConfig(
    static_dir=str(STATIC_DIR),
    raw_buildings_path=str(STATIC_DIR / 'no_raw_osm_for_virtual_case.json'),
    geometry_resolution_m=GRID_RESOLUTION_M,
    geometry_rotation_deg=0,
    model_resolution_m=GRID_RESOLUTION_M,
    target_xy=(Y_DIM, X_DIM),
    embed_pad_y=EMBED_PAD_Y,
    embed_pad_x=EMBED_PAD_X,
    z_dim=Z_DIM,
    height_scale_m=HEIGHT_SCALE_M,
    inlet_flow=INLET_FLOW,
    timesteppings=50,
    output_dir=str(VELOCITY_DIGIT_DIR),
)

temperature_config = Temperature3DScenarioConfig(
    output_dir=str(CASE_DIR),
    reuse_velocity_output_dir=str(VELOCITY_CACHE_DIR),
    frame_duration_s=90.0,
    diffusion_coeff_m2_s=1.0,
    velocity_scale=1.0,
    ambient_temp_c=24.2,
    inflow_temp_c=24.2,
    ambient_relative_humidity_pct=47.19,
    cloud_cover_fraction=0.47,
    reference_wind_speed_m_s=3.09,
    reference_wind_direction_deg=REFERENCE_WIND_DIRECTION_DEG,
    analysis_time_utc='2025-07-25T13:00:00+00:00',
    anthropogenic_heat_flux_w_m2=15.0,
    save_animation=False,
    save_overview_figure=False,
    temperature_solver_backend='auto',
    temperature_solver_device='cuda',
)

print(json.dumps({
    'velocity_config': velocity_config.__dict__,
    'temperature_config': temperature_config.__dict__,
}, indent=2))


## Run the Physical Label Generator



In [ ]:
run_outputs = run_temperature_3d_pipeline(
    velocity_config,
    temperature_config,
    keep_outputs=False,
)
summary = run_outputs['summary']
print(json.dumps({
    'case_dir': str(CASE_DIR),
    'velocity_cache_dir': str(VELOCITY_CACHE_DIR),
    'velocity_shape': summary.get('velocity_shape'),
    'temperature_shape': summary.get('temperature_shape'),
    'temperature_min_c': summary.get('temperature_min_c'),
    'temperature_max_c': summary.get('temperature_max_c'),
    'ground_surface_temperature_min_c': summary.get('ground_surface_temperature_min_c'),
    'ground_surface_temperature_max_c': summary.get('ground_surface_temperature_max_c'),
    'temperature_solver_backend_used': summary.get('temperature_solver_backend_used'),
    'cache_reused': summary.get('cache_reused'),
}, indent=2))


## Quick Check of the Generated Physical Fields

In [ ]:
temp_phys = np.load(CASE_DIR / 'temperature_fields_3d_masked_c.npy')
building_mask_2d = np.load(VELOCITY_CACHE_DIR / 'building_mask_2d.npy').astype(bool)
study_area_mask_2d = np.load(VELOCITY_CACHE_DIR / 'study_area_mask_2d.npy').astype(bool)
solid_mask_3d = np.load(VELOCITY_CACHE_DIR / 'solid_mask_3d.npy').astype(bool)
velocity_speed = np.load(VELOCITY_CACHE_DIR / 'velocity_speed_3d.npy', mmap_mode='r')

frame_id = temp_phys.shape[0] - 1
z_levels = [min(2, temp_phys.shape[1] - 1), min(6, temp_phys.shape[1] - 1)]
vmin = float(np.nanpercentile(temp_phys[frame_id], 2))
vmax = float(np.nanpercentile(temp_phys[frame_id], 98))

fig, axes = plt.subplots(len(z_levels), 2, figsize=(9, 4.0 * len(z_levels)), constrained_layout=True)
if len(z_levels) == 1:
    axes = axes[None, :]
for row, z_idx in enumerate(z_levels):
    temp_slice = temp_phys[frame_id, z_idx].copy()
    speed_slice = np.asarray(velocity_speed[min(frame_id, velocity_speed.shape[0] - 1), z_idx])
    temp_slice[~study_area_mask_2d] = np.nan
    speed_slice = speed_slice.copy()
    speed_slice[~study_area_mask_2d] = np.nan
    im0 = axes[row, 0].imshow(temp_slice, cmap='YlOrRd', vmin=vmin, vmax=vmax, origin='upper')
    axes[row, 0].contour(building_mask_2d.astype(float), levels=[0.5], colors='black', linewidths=0.5)
    axes[row, 0].set_title(f'Physical temperature | frame {frame_id} | z={z_idx}')
    im1 = axes[row, 1].imshow(speed_slice, cmap='viridis', origin='upper')
    axes[row, 1].contour(building_mask_2d.astype(float), levels=[0.5], colors='white', linewidths=0.5)
    axes[row, 1].set_title(f'DIGIT velocity speed | z={z_idx}')
    for ax in axes[row]:
        ax.set_xlabel('x index')
        ax.set_ylabel('y index')
fig.colorbar(im0, ax=axes[:, 0], shrink=0.85, label='deg C')
fig.colorbar(im1, ax=axes[:, 1], shrink=0.85, label='velocity magnitude')
plt.show()


## Load the Trained Surrogate

In [ ]:
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f'Checkpoint not found: {CHECKPOINT_PATH}')

checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu')
ckpt_config = checkpoint.get('config', {})
stats_payload = checkpoint.get('stats')
if stats_payload is None:
    stats_cache = RUN_DIR / 'surrogate_stats.json'
    stats_payload = json.loads(stats_cache.read_text())['stats']
stats = SurrogateStats(**stats_payload)

HISTORY_STEPS = int(ckpt_config.get('history_steps', 3))
PATCH_SIZE = tuple(ckpt_config.get('patch_size', (8, 32, 32)))
IN_CHANNELS = int(ckpt_config.get('in_channels', HISTORY_STEPS + 3 + 1 + 8))
BASE_CHANNELS = int(ckpt_config.get('base_channels', 12))
DEPTH = int(ckpt_config.get('depth', 3))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = UNet3D(
    in_channels=IN_CHANNELS,
    out_channels=1,
    base_channels=BASE_CHANNELS,
    depth=DEPTH,
    upsample_mode='trilinear',
).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print('Device:', device)
print('Loaded checkpoint epoch:', checkpoint.get('epoch', 'unknown'))
print('Model in_channels:', IN_CHANNELS)
print('Patch size:', PATCH_SIZE)


## Run Raw Surrogate Prediction on the Virtual Physical Case

In [ ]:
def make_full_volume_input(case, stats, history_steps):
    helper = TemperatureSurrogateDataset(
        [case],
        stats,
        history_steps=history_steps,
        patch_size=None,
        samples_per_case=1,
        random_crop=False,
        deterministic_sampling=True,
        seed=7,
    )
    t = case.n_training_steps - 1
    crop = (slice(None), slice(None), slice(None))
    temp_history = np.asarray(case.temperature[t - history_steps + 1 : t + 1], dtype=np.float32)
    u = np.asarray(case.u[t], dtype=np.float32)
    v = np.asarray(case.v[t], dtype=np.float32)
    w = np.asarray(case.w[t], dtype=np.float32)
    speed = np.sqrt(u * u + v * v + w * w)
    channels = [helper._norm_temp(frame) for frame in temp_history]
    channels.extend([u / stats.velocity_scale, v / stats.velocity_scale, w / stats.velocity_scale])
    if helper.include_speed:
        channels.append(speed / stats.velocity_scale)
    channels.extend(helper._static_channels(case, crop, t))
    x = np.stack(channels, axis=0).astype(np.float32)
    y_true = np.asarray(case.temperature[t + 1], dtype=np.float32)
    mask = helper._fluid_mask(case, crop).astype(bool)
    return x, y_true, mask, t + 1


def sliding_starts(size, tile, overlap):
    if tile >= size:
        return [0]
    stride = max(1, tile - overlap)
    starts = list(range(0, size - tile + 1, stride))
    if starts[-1] != size - tile:
        starts.append(size - tile)
    return starts


def predict_full_volume(case, model, device, stats, history_steps, tile_size=PATCH_SIZE, overlap=(2, 8, 8)):
    x_full, y_true, mask, target_frame = make_full_volume_input(case, stats, history_steps)
    _, zdim, ydim, xdim = x_full.shape
    tz, ty, tx = tile_size
    oz, oy, ox = overlap
    pred_sum = np.zeros((zdim, ydim, xdim), dtype=np.float32)
    pred_count = np.zeros((zdim, ydim, xdim), dtype=np.float32)
    z_starts = sliding_starts(zdim, tz, oz)
    y_starts = sliding_starts(ydim, ty, oy)
    x_starts = sliding_starts(xdim, tx, ox)
    with torch.no_grad():
        for z0 in z_starts:
            for y0 in y_starts:
                for x0 in x_starts:
                    tile = x_full[:, z0:z0+tz, y0:y0+ty, x0:x0+tx]
                    xt = torch.from_numpy(tile[None, ...]).to(device, non_blocking=True)
                    with torch.amp.autocast('cuda', enabled=device.type == 'cuda'):
                        pred_tile = model(xt)[0, 0]
                    pred_tile = (pred_tile.float().cpu().numpy() * stats.temp_std + stats.temp_mean).astype(np.float32)
                    pred_sum[z0:z0+tz, y0:y0+ty, x0:x0+tx] += pred_tile
                    pred_count[z0:z0+tz, y0:y0+ty, x0:x0+tx] += 1.0
    return pred_sum / np.maximum(pred_count, 1.0), y_true, mask, target_frame

virtual_case = Temperature3DCase(CASE_DIR)
pred_c, true_c, valid_mask, target_frame = predict_full_volume(
    virtual_case,
    model,
    device,
    stats,
    HISTORY_STEPS,
    tile_size=PATCH_SIZE,
    overlap=(2, 8, 8),
)
raw_err = pred_c - true_c
err_values = raw_err[valid_mask]
metrics = {
    'case_id': CASE_DIR.name,
    'target_frame': int(target_frame),
    'mean_bias_c': float(np.mean(err_values)),
    'mae_c': float(np.mean(np.abs(err_values))),
    'rmse_c': float(np.sqrt(np.mean(err_values ** 2))),
    'p95_abs_error_c': float(np.percentile(np.abs(err_values), 95)),
}
(CASE_DIR / 'surrogate_real_pipeline_virtual_test_metrics.json').write_text(json.dumps(metrics, indent=2))
print(json.dumps(metrics, indent=2))


## Compare Physical Model and Surrogate

In [ ]:
VIS_Z_LEVELS = [4, 11]

temp_cmap = plt.get_cmap('YlOrRd').copy()
temp_cmap.set_bad('white')
err_cmap = plt.get_cmap('coolwarm').copy()
err_cmap.set_bad('white')

plot_mask_3d = valid_mask.astype(bool)
valid_values = np.concatenate([true_c[plot_mask_3d], pred_c[plot_mask_3d]])
vmin = float(np.nanpercentile(valid_values, 2))
vmax = float(np.nanpercentile(valid_values, 98))
err_lim = float(np.nanpercentile(np.abs(raw_err[plot_mask_3d]), 98))

fig, axes = plt.subplots(len(VIS_Z_LEVELS), 3, figsize=(14, 4.2 * len(VIS_Z_LEVELS)), constrained_layout=True)
if len(VIS_Z_LEVELS) == 1:
    axes = axes[None, :]

for row, z_idx in enumerate(VIS_Z_LEVELS):
    z_idx = max(0, min(int(z_idx), true_c.shape[0] - 1))
    slice_valid = plot_mask_3d[z_idx]
    true_slice = true_c[z_idx].copy()
    pred_slice = pred_c[z_idx].copy()
    err_slice = raw_err[z_idx].copy()
    true_slice[~slice_valid] = np.nan
    pred_slice[~slice_valid] = np.nan
    err_slice[~slice_valid] = np.nan
    slice_bias = float(np.mean(raw_err[z_idx][slice_valid])) if np.any(slice_valid) else np.nan
    slice_mae = float(np.mean(np.abs(raw_err[z_idx][slice_valid]))) if np.any(slice_valid) else np.nan

    im0 = axes[row, 0].imshow(true_slice, cmap=temp_cmap, vmin=vmin, vmax=vmax, origin='upper')
    axes[row, 0].set_title(f'Physical model T\nframe {target_frame} | z={z_idx}')
    im1 = axes[row, 1].imshow(pred_slice, cmap=temp_cmap, vmin=vmin, vmax=vmax, origin='upper')
    axes[row, 1].set_title(f'Surrogate prediction T\nraw output | z={z_idx}')
    im2 = axes[row, 2].imshow(err_slice, cmap=err_cmap, vmin=-err_lim, vmax=err_lim, origin='upper')
    axes[row, 2].set_title(f'Raw error: surrogate - physical model\nbias={slice_bias:+.3f} C | MAE={slice_mae:.3f} C')
    for ax in axes[row]:
        ax.contour(building_mask_2d.astype(float), levels=[0.5], colors='black', linewidths=0.5, alpha=0.75)
        ax.set_xlabel('x index')
        ax.set_ylabel('y index')
        ax.tick_params(labelsize=8)

fig.colorbar(im1, ax=axes[:, :2], shrink=0.85, fraction=0.025, pad=0.02, label='deg C')
fig.colorbar(im2, ax=axes[:, 2], shrink=0.85, fraction=0.025, pad=0.02, label='deg C')
fig.suptitle('Real-pipeline virtual geometry test | raw surrogate output | no correction', fontsize=14)
plt.show()


## GIF Animations: Horizontal and Vertical Sections

This section generates GIF animations for the real-pipeline virtual geometry test and displays them in the notebook.

The animations compare physical temperature, raw surrogate prediction, and raw error. To keep runtime and file size reasonable, the default uses every second available frame. Set `GIF_FRAME_STRIDE = 1` for all frames.


In [ ]:
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image as IPyImage, display

GIF_DIR = CASE_DIR / 'gif_comparisons'
GIF_DIR.mkdir(parents=True, exist_ok=True)

GIF_FRAME_STRIDE = 2
GIF_FPS = 4
GIF_Z_INDEX = min(2, temp_phys.shape[1] - 1)
GIF_X_SECTION_INDEX = temp_phys.shape[3] // 2

# Frames correspond to one-step prediction targets. The first valid target is HISTORY_STEPS.
frame_ids = list(range(HISTORY_STEPS, virtual_case.temperature.shape[0], GIF_FRAME_STRIDE))
print('GIF output dir:', GIF_DIR)
print('Animation frames:', frame_ids[:5], '...', frame_ids[-5:], '| count:', len(frame_ids))


def make_full_volume_input_at_t(case, stats, history_steps, t):
    helper = TemperatureSurrogateDataset(
        [case],
        stats,
        history_steps=history_steps,
        patch_size=None,
        samples_per_case=1,
        random_crop=False,
        deterministic_sampling=True,
        seed=7,
    )
    crop = (slice(None), slice(None), slice(None))
    temp_history = np.asarray(case.temperature[t - history_steps + 1 : t + 1], dtype=np.float32)
    velocity_t = min(t, case.u.shape[0] - 1)
    u = np.asarray(case.u[velocity_t], dtype=np.float32)
    v = np.asarray(case.v[velocity_t], dtype=np.float32)
    w = np.asarray(case.w[velocity_t], dtype=np.float32)
    speed = np.sqrt(u * u + v * v + w * w)
    channels = [helper._norm_temp(frame) for frame in temp_history]
    channels.extend([u / stats.velocity_scale, v / stats.velocity_scale, w / stats.velocity_scale])
    if helper.include_speed:
        channels.append(speed / stats.velocity_scale)
    channels.extend(helper._static_channels(case, crop, t))
    x = np.stack(channels, axis=0).astype(np.float32)
    y_true = np.asarray(case.temperature[t + 1], dtype=np.float32)
    mask = helper._fluid_mask(case, crop).astype(bool)
    return x, y_true, mask


def predict_full_volume_at_t(case, model, device, stats, history_steps, target_frame, tile_size=PATCH_SIZE, overlap=(2, 8, 8)):
    t = target_frame - 1
    x_full, y_true, mask = make_full_volume_input_at_t(case, stats, history_steps, t)
    _, zdim, ydim, xdim = x_full.shape
    tz, ty, tx = tile_size
    oz, oy, ox = overlap
    pred_sum = np.zeros((zdim, ydim, xdim), dtype=np.float32)
    pred_count = np.zeros((zdim, ydim, xdim), dtype=np.float32)
    z_starts = sliding_starts(zdim, tz, oz)
    y_starts = sliding_starts(ydim, ty, oy)
    x_starts = sliding_starts(xdim, tx, ox)
    with torch.no_grad():
        for z0 in z_starts:
            for y0 in y_starts:
                for x0 in x_starts:
                    tile = x_full[:, z0:z0+tz, y0:y0+ty, x0:x0+tx]
                    xt = torch.from_numpy(tile[None, ...]).to(device, non_blocking=True)
                    with torch.amp.autocast('cuda', enabled=device.type == 'cuda'):
                        pred_tile = model(xt)[0, 0]
                    pred_tile = (pred_tile.float().cpu().numpy() * stats.temp_std + stats.temp_mean).astype(np.float32)
                    pred_sum[z0:z0+tz, y0:y0+ty, x0:x0+tx] += pred_tile
                    pred_count[z0:z0+tz, y0:y0+ty, x0:x0+tx] += 1.0
    return pred_sum / np.maximum(pred_count, 1.0), y_true, mask

pred_seq = []
true_seq = []
mask_seq = []
for idx, frame_id in enumerate(frame_ids, start=1):
    print(f'[gif inference {idx}/{len(frame_ids)}] frame {frame_id}', flush=True)
    pred_frame, true_frame, mask_frame = predict_full_volume_at_t(
        virtual_case,
        model,
        device,
        stats,
        HISTORY_STEPS,
        target_frame=frame_id,
        tile_size=PATCH_SIZE,
        overlap=(2, 8, 8),
    )
    pred_seq.append(pred_frame.astype(np.float32))
    true_seq.append(true_frame.astype(np.float32))
    mask_seq.append(mask_frame.astype(bool))

pred_seq = np.asarray(pred_seq, dtype=np.float32)
true_seq = np.asarray(true_seq, dtype=np.float32)
mask_seq = np.asarray(mask_seq, dtype=bool)
err_seq = pred_seq - true_seq

valid_values = np.concatenate([true_seq[mask_seq], pred_seq[mask_seq]])
anim_vmin = float(np.nanpercentile(valid_values, 2))
anim_vmax = float(np.nanpercentile(valid_values, 98))
anim_err_lim = float(np.nanpercentile(np.abs(err_seq[mask_seq]), 98))

print('Animation temperature color range:', (anim_vmin, anim_vmax))
print('Animation error color range:', (-anim_err_lim, anim_err_lim))


### Horizontal Slice GIF

This GIF shows a fixed horizontal `z` slice through time.


In [ ]:
def mask_2d(arr2d, mask2d):
    out = arr2d.copy()
    out[~mask2d] = np.nan
    return out

z_idx = max(0, min(int(GIF_Z_INDEX), true_seq.shape[1] - 1))
fig, axes = plt.subplots(1, 3, figsize=(14, 4.6), constrained_layout=True)

initial_mask = mask_seq[0, z_idx]
im0 = axes[0].imshow(mask_2d(true_seq[0, z_idx], initial_mask), cmap=temp_cmap, vmin=anim_vmin, vmax=anim_vmax, origin='upper')
im1 = axes[1].imshow(mask_2d(pred_seq[0, z_idx], initial_mask), cmap=temp_cmap, vmin=anim_vmin, vmax=anim_vmax, origin='upper')
im2 = axes[2].imshow(mask_2d(err_seq[0, z_idx], initial_mask), cmap=err_cmap, vmin=-anim_err_lim, vmax=anim_err_lim, origin='upper')

for ax in axes:
    ax.contour(building_mask_2d.astype(float), levels=[0.5], colors='black', linewidths=0.45, alpha=0.75)
    ax.set_xlabel('x index')
    ax.set_ylabel('y index')
    ax.tick_params(labelsize=8)

fig.colorbar(im1, ax=axes[:2], shrink=0.85, fraction=0.03, pad=0.02, label='deg C')
fig.colorbar(im2, ax=axes[2], shrink=0.85, fraction=0.03, pad=0.02, label='deg C')


def update_horizontal(frame_idx):
    frame_id = frame_ids[frame_idx]
    slice_mask = mask_seq[frame_idx, z_idx]
    im0.set_data(mask_2d(true_seq[frame_idx, z_idx], slice_mask))
    im1.set_data(mask_2d(pred_seq[frame_idx, z_idx], slice_mask))
    im2.set_data(mask_2d(err_seq[frame_idx, z_idx], slice_mask))
    if np.any(slice_mask):
        bias = float(np.mean(err_seq[frame_idx, z_idx][slice_mask]))
        mae = float(np.mean(np.abs(err_seq[frame_idx, z_idx][slice_mask])))
    else:
        bias = np.nan
        mae = np.nan
    axes[0].set_title(f'Physical model T\nframe {frame_id} | z={z_idx}')
    axes[1].set_title(f'Surrogate prediction T\nraw output | z={z_idx}')
    axes[2].set_title(f'Raw error: surrogate - physical\nbias={bias:+.3f} C | MAE={mae:.3f} C')
    return im0, im1, im2

anim = FuncAnimation(fig, update_horizontal, frames=len(frame_ids), interval=1000 / GIF_FPS, blit=False)
horizontal_gif_path = GIF_DIR / f'horizontal_z{z_idx:02d}_physical_vs_surrogate.gif'
anim.save(horizontal_gif_path, writer=PillowWriter(fps=GIF_FPS), dpi=120)
plt.close(fig)
print('Saved horizontal GIF:', horizontal_gif_path)
display(IPyImage(filename=str(horizontal_gif_path)))


### Vertical Section GIF

This GIF shows an `x-z` vertical section at a fixed `x` index through time.


In [ ]:
# Vertical x-z GIF: fixed y section.
# This is usually better than fixed x if the dominant transport is along x.

GIF_Y_SECTION_INDEX = None  # None means use the domain center.

if GIF_Y_SECTION_INDEX is None:
    y_idx = true_seq.shape[2] // 2
else:
    y_idx = max(0, min(int(GIF_Y_SECTION_INDEX), true_seq.shape[2] - 1))

fig, axes = plt.subplots(1, 3, figsize=(14, 4.4), constrained_layout=True)

# For vertical sections, show z increasing upward by using origin='lower'.
# Fixed y gives x-z section: array shape is z by x.
initial_mask_v = mask_seq[0, :, y_idx, :]

im0 = axes[0].imshow(
    mask_2d(true_seq[0, :, y_idx, :], initial_mask_v),
    cmap=temp_cmap,
    vmin=anim_vmin,
    vmax=anim_vmax,
    origin='lower',
    aspect='auto',
)
im1 = axes[1].imshow(
    mask_2d(pred_seq[0, :, y_idx, :], initial_mask_v),
    cmap=temp_cmap,
    vmin=anim_vmin,
    vmax=anim_vmax,
    origin='lower',
    aspect='auto',
)
im2 = axes[2].imshow(
    mask_2d(err_seq[0, :, y_idx, :], initial_mask_v),
    cmap=err_cmap,
    vmin=-anim_err_lim,
    vmax=anim_err_lim,
    origin='lower',
    aspect='auto',
)

for ax in axes:
    ax.set_xlabel('x index')
    ax.set_ylabel('z index')
    ax.tick_params(labelsize=8)

fig.colorbar(im1, ax=axes[:2], shrink=0.85, fraction=0.03, pad=0.02, label='deg C')
fig.colorbar(im2, ax=axes[2], shrink=0.85, fraction=0.03, pad=0.02, label='deg C')


def update_vertical(frame_idx):
    frame_id = frame_ids[frame_idx]
    section_mask = mask_seq[frame_idx, :, y_idx, :]

    im0.set_data(mask_2d(true_seq[frame_idx, :, y_idx, :], section_mask))
    im1.set_data(mask_2d(pred_seq[frame_idx, :, y_idx, :], section_mask))
    im2.set_data(mask_2d(err_seq[frame_idx, :, y_idx, :], section_mask))

    if np.any(section_mask):
        section_err = err_seq[frame_idx, :, y_idx, :][section_mask]
        bias = float(np.mean(section_err))
        mae = float(np.mean(np.abs(section_err)))
    else:
        bias = np.nan
        mae = np.nan

    axes[0].set_title(f'Physical model T\nframe {frame_id} | y={y_idx}')
    axes[1].set_title(f'Surrogate prediction T\nraw output | y={y_idx}')
    axes[2].set_title(f'Raw error: surrogate - physical\nbias={bias:+.3f} C | MAE={mae:.3f} C')

    return im0, im1, im2


anim = FuncAnimation(
    fig,
    update_vertical,
    frames=len(frame_ids),
    interval=1000 / GIF_FPS,
    blit=False,
)

vertical_gif_path = GIF_DIR / f'vertical_y{y_idx:03d}_xz_physical_vs_surrogate.gif'
anim.save(vertical_gif_path, writer=PillowWriter(fps=GIF_FPS), dpi=120)
plt.close(fig)

print('Saved vertical GIF:', vertical_gif_path)
display(IPyImage(filename=str(vertical_gif_path)))

In [ ]:
# Generate velocity GIFs: horizontal z-slice and vertical x-z section.
# Put this cell after the physical pipeline has finished.

from pathlib import Path
from IPython.display import Image, display
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

VELOCITY_GIF_DIR = CASE_DIR / "gif_comparisons"
VELOCITY_GIF_DIR.mkdir(parents=True, exist_ok=True)

VELOCITY_GIF_FRAME_STRIDE = 2
VELOCITY_GIF_FPS = 4

# Change these values if needed.
VELOCITY_GIF_Z_INDEX = min(2, Z_DIM - 1)

# None means use the domain center.
# This controls the vertical x-z section position.
VELOCITY_GIF_Y_SECTION_INDEX = None

u = np.load(VELOCITY_CACHE_DIR / "velocity_u_3d.npy")
v = np.load(VELOCITY_CACHE_DIR / "velocity_v_3d.npy")
w = np.load(VELOCITY_CACHE_DIR / "velocity_w_3d.npy")
speed = np.load(VELOCITY_CACHE_DIR / "velocity_speed_3d.npy")
solid_mask = np.load(VELOCITY_CACHE_DIR / "solid_mask_3d.npy").astype(bool)

if VELOCITY_GIF_Y_SECTION_INDEX is None:
    VELOCITY_GIF_Y_SECTION_INDEX = speed.shape[2] // 2

frame_ids = list(range(0, speed.shape[0], VELOCITY_GIF_FRAME_STRIDE))

if solid_mask.ndim == 3 and speed.ndim == 4:
    solid_mask_time = np.broadcast_to(solid_mask[None, ...], speed.shape)
else:
    solid_mask_time = solid_mask

speed_plot = np.ma.array(speed, mask=solid_mask_time)
speed_cmap = plt.get_cmap("viridis").copy()
speed_cmap.set_bad("white")

vmin = float(np.nanpercentile(speed_plot.compressed(), 2))
vmax = float(np.nanpercentile(speed_plot.compressed(), 98))

print("Velocity field shape:", speed.shape)
print("Frames:", frame_ids[0], "to", frame_ids[-1], "| total gif frames:", len(frame_ids))
print("Horizontal z index:", VELOCITY_GIF_Z_INDEX)
print("Vertical x-z section y index:", VELOCITY_GIF_Y_SECTION_INDEX)


def save_velocity_horizontal_gif():
    fig, ax = plt.subplots(figsize=(5.4, 4.8))
    first = frame_ids[0]

    img = ax.imshow(
        speed_plot[first, VELOCITY_GIF_Z_INDEX],
        cmap=speed_cmap,
        vmin=vmin,
        vmax=vmax,
        origin="upper",
    )
    cb = fig.colorbar(img, ax=ax, fraction=0.035, pad=0.025)
    cb.set_label("m/s")

    ax.set_xlabel("x index")
    ax.set_ylabel("y index")
    ax.set_title(f"Velocity speed | frame {first} | z={VELOCITY_GIF_Z_INDEX}")

    def update(frame_id):
        img.set_data(speed_plot[frame_id, VELOCITY_GIF_Z_INDEX])
        ax.set_title(f"Velocity speed | frame {frame_id} | z={VELOCITY_GIF_Z_INDEX}")
        return [img]

    ani = animation.FuncAnimation(
        fig,
        update,
        frames=frame_ids,
        interval=1000 / VELOCITY_GIF_FPS,
        blit=False,
    )

    out_path = VELOCITY_GIF_DIR / f"velocity_horizontal_z{VELOCITY_GIF_Z_INDEX:02d}.gif"
    ani.save(out_path, writer="pillow", fps=VELOCITY_GIF_FPS)
    plt.close(fig)
    return out_path


def save_velocity_vertical_gif():
    fig, ax = plt.subplots(figsize=(6.8, 3.6))
    first = frame_ids[0]

    # Vertical section at fixed y: shape is z by x.
    # This shows the x-z plane and is usually better for seeing along-wind transport.
    section = speed_plot[first, :, VELOCITY_GIF_Y_SECTION_INDEX, :]

    img = ax.imshow(
        section,
        cmap=speed_cmap,
        vmin=vmin,
        vmax=vmax,
        origin="lower",
        aspect="auto",
    )
    cb = fig.colorbar(img, ax=ax, fraction=0.028, pad=0.02)
    cb.set_label("m/s")

    ax.set_xlabel("x index")
    ax.set_ylabel("z index")
    ax.set_title(
        f"Velocity speed vertical x-z section | frame {first} | y={VELOCITY_GIF_Y_SECTION_INDEX}"
    )

    def update(frame_id):
        img.set_data(speed_plot[frame_id, :, VELOCITY_GIF_Y_SECTION_INDEX, :])
        ax.set_title(
            f"Velocity speed vertical x-z section | frame {frame_id} | y={VELOCITY_GIF_Y_SECTION_INDEX}"
        )
        return [img]

    ani = animation.FuncAnimation(
        fig,
        update,
        frames=frame_ids,
        interval=1000 / VELOCITY_GIF_FPS,
        blit=False,
    )

    out_path = VELOCITY_GIF_DIR / f"velocity_vertical_y{VELOCITY_GIF_Y_SECTION_INDEX:03d}_xz.gif"
    ani.save(out_path, writer="pillow", fps=VELOCITY_GIF_FPS)
    plt.close(fig)
    return out_path


horizontal_gif = save_velocity_horizontal_gif()
vertical_gif = save_velocity_vertical_gif()

print("Saved horizontal velocity GIF:", horizontal_gif)
display(Image(filename=str(horizontal_gif)))

print("Saved vertical velocity GIF:", vertical_gif)
display(Image(filename=str(vertical_gif)))